## Assessment 2

Use dataset from Assessment 1 to train a machine learning model on historical records then appluy it to a simulated real-time stream. This assessment supports unit learning outcomes 3,4 and 5. 

### Before Part A
Data is split into training dataset, 70% of records, to train and evaluate the ML model and remaining 30% will be used for streaming dataset and will be replayed through Kfka in Part B.

In [6]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

#setup a spark session and load dataset
master = "local[*]"
app_name = "Uber Business Query DF"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

sc = spark.sparkContext

df = spark.read.csv("ncr_ride_bookings.csv",header=True)

#repeat of assessment one to ensure data type is correct
from pyspark.sql.functions import col,concat_ws
from pyspark.sql.types import DateType

# Replace string 'null' with true None/null globally
df = df.replace("null", None)

float_col = ["Avg VTAT","Avg CTAT","Ride Distance","Driver Ratings","Customer Ratings","Booking Value"]
df = df.select([col(c).cast("float") if c in float_col else col(c) for c in df.columns])
df = df.withColumn("Datetime",concat_ws(" ", col("Date"), col("Time")).cast("timestamp"))
df = df.drop("Date", "Time")



In [9]:
train_df, stream_df = df.randomSplit([0.7,0.3],seed = 42)

#to ensure a 70/30 split of data
print(train_df.count())
print(train_df.count()/150000)
print(stream_df.count())
print(stream_df.count()/150000)

#remove all entries that do no have a driver rating number, which is the number the customer gives to the driver
#this is done after the initial split to reflect what real-time data would be, as the driver rating is optional for customers
train_df = train_df.dropna(subset=["Driver Ratings"])
print(train_df.count())

105209
0.7013933333333333
44791
0.2986066666666667
65230


### Part A - The ML Pipeline

#### 1. The ML task selection

The ideal machine learning algorithm for the original business case of trying to predict the driver rating of their ride based on the record features would be regression. This is becuase the ratings are numerical in value, continuous after some data cleaning, and do have meaningful variance. The dataset included will only included records that do have a driver rating, which is much lower than the original dataset but still large enough to create a model with meaningful result. 

#### 2. Feature Enginerring

For this dataset, there are four general types of data that is included - categorical, numerical and free text. Free text is seperated as it required further work to categorise this data. 

In [10]:
train_df.show(5)

+----------------+--------------+----------------+-------------+---------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+-------------------+
|      Booking ID|Booking Status|     Customer ID| Vehicle Type|Pickup Location|   Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|           Datetime|
+----------------+--------------+----------------+-------------+---------------+----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+---------------------

In [17]:
#seperate numerically relevant columns from category columns as well as binary columns and free text
categoryInputCols = ["Booking Status", "Vechile Type", "Pickup Location", "Drop Location","Payment Method", "Cancelled Rides by Customer", 
"Cancelled Rides by Driver", "Incomplete Rides"]
freetextInputCols = ["Reason for cancelling by Customer","Driver Cancellation Reason","Incomplete Rides Reason"]
numericInputCols = ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Customer Ratings", "Datetime"]
numericOutputCol = "Driver Ratings"

numericalCols = [numericOutputCol]+numericInputCols